# VietMed-NER — Frozen-Encoder Head Ablation

Compare three decoding heads — **Softmax · CRF · GAT+CRF** — on top of a **frozen** encoder
(default **ViHealthBERT**, a domain-pretrained Vietnamese medical model), using **cached
features** (forward the encoder once). Fair comparison: all heads share the same frozen
features; only the head differs.

Two no-/low-cost performance levers are built in:
- **Domain encoder (ViHealthBERT) frozen** — better medical features than vanilla PhoBERT, no
  encoder training (swap `CFG["encoder"]` back to `vinai/phobert-base` to compare).
- **BIO-constrained CRF** — illegal tag transitions (`O→I`, `B-X→I-Y`, start with `I-`) are
  pinned to `-1e4` and frozen, so the CRF/GAT+CRF heads can never emit BIO-invalid spans.

Spec: `docs/superpowers/specs/2026-06-30-frozen-encoder-head-ablation-design.md`
Plan: `docs/superpowers/plans/2026-06-30-frozen-encoder-head-ablation-plan.md`

> **Claim (honest):** a richer graph-attention + CRF head beats simpler heads *on the same frozen
> representations* — a head ablation, **not** a syntactic-dependency-graph claim (the GAT here is
> fully-connected, PAD-masked).


In [ ]:
# §0 Setup
!pip -q install "transformers==4.44.2" "datasets==2.21.0" seqeval pytorch-crf py_vncorenlp pandas
!pip -q install torch_geometric

import os, random, numpy as np, torch, torch.nn as nn, torch.nn.functional as F

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(42)

from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR   = '/content/drive/MyDrive/vietmed_ner'
RESULTS_DIR = f'{DRIVE_DIR}/results'
CKPT_DIR    = f'{DRIVE_DIR}/checkpoints'
CACHE_DIR   = f'{DRIVE_DIR}/feat_cache'
for d in (RESULTS_DIR, CKPT_DIR, CACHE_DIR):
    os.makedirs(d, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

In [ ]:
# §1 Config
# encoder: ViHealthBERT (domain) by default; swap to "vinai/phobert-base" to compare.
CFG   = {"encoder": "demdecuong/vihealthbert-base-word", "max_len": 128, "segment": True, "batch": 32}
SEEDS = [42, 1, 2]

In [ ]:
# §2 Data  (reused from notebook 05)
from datasets import load_dataset
raw = load_dataset("leduckhai/VietMed-NER")
for c in ("audio", "duration"):
    if c in raw["train"].column_names:
        raw = raw.remove_columns(c)

TAG_COL  = "labels" if "labels" in raw["train"].column_names else "tags"
WORD_COL = "words"

def get_words_labels(split):
    ds = raw[split]
    return list(ds[WORD_COL]), list(ds[TAG_COL])

_all_tags = set(t for split in raw for row in raw[split][TAG_COL] for t in row)
LABEL_LIST = ["O"] + sorted(t for t in _all_tags if t != "O")
label2id = {t:i for i,t in enumerate(LABEL_LIST)}
id2label = {i:t for t,i in label2id.items()}
print(len(LABEL_LIST), "labels; splits:", {k: len(raw[k]) for k in raw})

In [ ]:
# §3 Preprocess: syllable->word tag re-alignment  (reused, verified on CPU)
def realign_tags(syllables, tags, word_groups):
    words, new_tags = [], []
    for grp in word_groups:
        words.append("_".join(syllables[i] for i in grp))
        head = tags[grp[0]]
        if head.startswith("I-"):                 # word boundary fell inside an entity
            head = "B-" + head[2:]
        new_tags.append(head)
    return words, new_tags

In [ ]:
# §3 (cont): py_vncorenlp word segmentation -> word_groups  (reused)
import py_vncorenlp
_SEG = None
def get_segmenter():
    global _SEG
    if _SEG is None:
        os.makedirs(f'{DRIVE_DIR}/vncorenlp', exist_ok=True)
        py_vncorenlp.download_model(save_dir=f'{DRIVE_DIR}/vncorenlp')
        _SEG = py_vncorenlp.VnCoreNLP(annotators=["wseg"], save_dir=f'{DRIVE_DIR}/vncorenlp')
    return _SEG

def segment_to_groups(syllables):
    """Return word_groups: list of syllable-index lists. Fallback: 1 syllable/word."""
    seg = get_segmenter()
    segged = seg.word_segment(" ".join(syllables))
    word_units = " ".join(segged).split()
    groups, cur = [], 0
    for wu in word_units:
        n = wu.count("_") + 1
        groups.append(list(range(cur, cur + n)))
        cur += n
    if cur != len(syllables):
        return [[i] for i in range(len(syllables))]
    return groups

In [ ]:
# §3 (cont): subword -100 alignment + build_encoded  (reused)
def align_labels(tokenized_inputs, word_label_ids):
    aligned = []
    for i, labels in enumerate(word_label_ids):
        prev, ids = None, []
        for wid in tokenized_inputs.word_ids(batch_index=i):
            if wid is None:
                ids.append(-100)
            elif wid != prev:
                ids.append(labels[wid])
            else:
                ids.append(-100)
            prev = wid
        aligned.append(ids)
    tokenized_inputs["labels"] = aligned
    return tokenized_inputs

def build_encoded(split, cfg, tokenizer):
    words_col, tags_col = get_words_labels(split)
    enc_words, enc_label_ids = [], []
    for syl, tags in zip(words_col, tags_col):
        if cfg["segment"]:
            groups = segment_to_groups(syl)
            w, t = realign_tags(syl, tags, groups)
        else:
            w, t = syl, tags
        enc_words.append(w)
        enc_label_ids.append([label2id[x] for x in t])
    tok = tokenizer(
        enc_words,
        is_split_into_words=True,
        truncation=True,
        max_length=cfg["max_len"],
        padding="max_length",
    )
    tok = align_labels(tok, enc_label_ids)
    return tok

In [ ]:
# §T tests — CPU-only, pure functions actually used by this notebook
def _test_realign():
    syl=["bệnh","nhân","bị","viêm","phổi"]
    tags=["B-DISEASESYMPTOM","I-DISEASESYMPTOM","O","B-DISEASESYMPTOM","I-DISEASESYMPTOM"]
    w,t = realign_tags(syl, tags, [[0,1],[2],[3,4]])
    assert w == ["bệnh_nhân","bị","viêm_phổi"] and t == ["B-DISEASESYMPTOM","O","B-DISEASESYMPTOM"]
    assert realign_tags(["a","b"], ["O","I-AGE"], [[0],[1]])[1] == ["O","B-AGE"]
    print("realign_tags OK")

class _FakeTok(dict):
    def __init__(self, wid): super().__init__(); self._wid=wid
    def word_ids(self, batch_index=0): return self._wid[batch_index]
def _test_align():
    out = align_labels(_FakeTok([[None,0,0,1,None]]), [[5,7]])
    assert out["labels"][0] == [-100,5,-100,7,-100]
    print("align_labels OK")

_test_realign(); _test_align()
print("ALL §T TESTS PASSED")

In [ ]:
# §4 Verify (GATE) — must PASS before caching/training
from transformers import RobertaTokenizerFast

def verify_roundtrip(cfg, tokenizer, n=5):
    tok = build_encoded("train", cfg, tokenizer)
    words_col, tags_col = get_words_labels("train")
    ok = 0
    for i in range(n):
        wids = tok.word_ids(batch_index=i)
        labs = tok["labels"][i]
        rec, seen = [], set()
        for wid, lab in zip(wids, labs):
            if wid is None or wid in seen:
                continue
            seen.add(wid)
            rec.append(id2label[lab])
        syl, tags = words_col[i], tags_col[i]
        exp = realign_tags(syl, tags, segment_to_groups(syl))[1] if cfg["segment"] else tags
        exp = exp[:len(rec)]
        assert rec == exp, f"sample {i} mismatch:\n  rec={rec}\n  exp={exp}"
        ok += 1
    print(f"verify_roundtrip PASSED for {ok}/{n} samples")

tokenizer = RobertaTokenizerFast.from_pretrained(CFG["encoder"], add_prefix_space=True)
verify_roundtrip(CFG, tokenizer)

In [ ]:
# §5 Feature cache — freeze PhoBERT, forward ONCE, store fp16 features
from transformers import AutoModel

def load_frozen_encoder(tokenizer):
    enc = AutoModel.from_pretrained(CFG["encoder"])
    enc.resize_token_embeddings(len(tokenizer))   # parity with tokenizer vocab
    enc = enc.to(DEVICE).eval()
    for p in enc.parameters():
        p.requires_grad = False
    return enc

def cache_features(split, tokenizer, encoder, batch=32):
    enc_tag = CFG["encoder"].split("/")[-1]          # keep caches per-encoder (no collision)
    path = f'{CACHE_DIR}/cache_{enc_tag}_{split}.pt'
    if os.path.exists(path):
        print(f'{split}: load cached {path}')
        return torch.load(path)
    tok  = build_encoded(split, CFG, tokenizer)
    ids  = torch.tensor(tok["input_ids"]); mask = torch.tensor(tok["attention_mask"])
    labs = torch.tensor(tok["labels"])
    feats = []
    with torch.no_grad():
        for i in range(0, len(ids), batch):
            h = encoder(ids[i:i+batch].to(DEVICE),
                        attention_mask=mask[i:i+batch].to(DEVICE)).last_hidden_state
            feats.append(h.half().cpu())
    feats = torch.cat(feats, 0)
    assert feats.shape[:2] == labs.shape, (feats.shape, labs.shape)
    obj = {"features": feats, "mask": mask, "labels": labs}
    torch.save(obj, path)
    print(f'{split}: {tuple(feats.shape)} cached -> {path}')
    return obj

encoder = load_frozen_encoder(tokenizer)
CACHE = {s: cache_features(s, tokenizer, encoder, CFG["batch"])
         for s in ["train", "validation", "test"]}
del encoder
if DEVICE == "cuda": torch.cuda.empty_cache()

In [ ]:
# §6 Heads — operate on cached features [B, L, H]; never touch PhoBERT
from torchcrf import CRF
from torch_geometric.nn import GATConv
from torch_geometric.data import Data, Batch

C = len(LABEL_LIST)
H = CACHE["train"]["features"].shape[-1]   # hidden size from cache (encoder-agnostic)

def _bio_masks(label_list):
    """Boolean masks of BIO-illegal transitions and illegal start tags."""
    n = len(label_list)
    def parse(t): return ("O", None) if t == "O" else (t[0], t[2:])
    bad_trans = torch.zeros(n, n, dtype=torch.bool)
    bad_start = torch.zeros(n, dtype=torch.bool)
    for j, tj in enumerate(label_list):
        if parse(tj)[0] == "I":
            bad_start[j] = True                          # a sequence cannot start with I-
    for i, ti in enumerate(label_list):
        pi, tyi = parse(ti)
        for j, tj in enumerate(label_list):
            pj, tyj = parse(tj)
            if pj == "I" and not (pi in ("B", "I") and tyi == tyj):
                bad_trans[i, j] = True                   # I-Y only allowed after B-Y or I-Y
    return bad_trans, bad_start

def bio_constrain(crf, label_list, neg=-1e4):
    """Pin BIO-illegal CRF transitions to `neg` and freeze them (grad masked to 0)."""
    bad_trans, bad_start = _bio_masks(label_list)
    with torch.no_grad():
        crf.transitions[bad_trans] = neg
        crf.start_transitions[bad_start] = neg
    crf.transitions.register_hook(lambda g: g.masked_fill(bad_trans.to(g.device), 0.0))
    crf.start_transitions.register_hook(lambda g: g.masked_fill(bad_start.to(g.device), 0.0))

class SoftmaxHead(nn.Module):
    head_kind = "softmax"   # note: greedy/independent, no BIO constraint applicable
    def __init__(self, H, C):
        super().__init__(); self.fc = nn.Linear(H, C)
    def forward(self, feats, mask, labels=None):
        logits = self.fc(feats)
        if labels is not None:
            return F.cross_entropy(logits.reshape(-1, C), labels.reshape(-1), ignore_index=-100)
        return logits.argmax(-1)                      # [B, L]

class CrfHead(nn.Module):
    head_kind = "crf"
    def __init__(self, H, C):
        super().__init__(); self.fc = nn.Linear(H, C); self.crf = CRF(C, batch_first=True)
        bio_constrain(self.crf, LABEL_LIST)           # BIO-constrained decoding
    def forward(self, feats, mask, labels=None):
        em = self.fc(feats); m = mask.bool()
        if labels is not None:
            safe = labels.clone(); safe[safe == -100] = 0
            return -self.crf(em, safe, mask=m, reduction="mean")
        return self.crf.decode(em, mask=m)            # list[list[int]]

class GatCrfHead(nn.Module):
    head_kind = "crf"
    def __init__(self, H, C, heads=4):
        super().__init__()
        self.gat = GATConv(H, H // heads, heads=heads, dropout=0.1)
        self.fc  = nn.Linear(H, C)
        self.crf = CRF(C, batch_first=True)
        bio_constrain(self.crf, LABEL_LIST)           # BIO-constrained decoding
    def _emissions(self, feats, mask):
        B, L, _ = feats.shape
        datas = []
        for b in range(B):
            n = int(mask[b].sum()); x = feats[b, :n]          # real tokens only (PAD masked out)
            idx = torch.arange(n, device=feats.device)
            ei  = torch.stack([idx.repeat_interleave(n), idx.repeat(n)], 0)  # FC over real tokens
            datas.append(Data(x=x, edge_index=ei))
        g   = Batch.from_data_list(datas)
        out = self.gat(g.x, g.edge_index)                     # [sum_n, H]
        h, ptr = feats.clone(), 0
        for b in range(B):
            n = int(mask[b].sum()); h[b, :n] = out[ptr:ptr+n]; ptr += n
        return self.fc(h)
    def forward(self, feats, mask, labels=None):
        em = self._emissions(feats, mask); m = mask.bool()
        if labels is not None:
            safe = labels.clone(); safe[safe == -100] = 0
            return -self.crf(em, safe, mask=m, reduction="mean")
        return self.crf.decode(em, mask=m)

In [ ]:
# §7 Eval helpers  (reused from notebook 05)
from seqeval.metrics import f1_score, classification_report
def evaluate_tags(true_tags, pred_tags):
    return {
       "micro_f1": f1_score(true_tags, pred_tags, average="micro"),
       "macro_f1": f1_score(true_tags, pred_tags, average="macro"),
       "report":   classification_report(true_tags, pred_tags, digits=4),
    }

def decode_predictions(pred, label_ids, head):
    true_tags, pred_tags = [], []
    for i, gold in enumerate(label_ids):
        p_seq = pred[i]
        t_row, p_row, j = [], [], 0
        for k, g in enumerate(gold):
            if g == -100:
                continue
            t_row.append(id2label[g])
            p_row.append(id2label[p_seq[k]] if head == "softmax" else id2label[p_seq[j]])
            j += 1
        true_tags.append(t_row); pred_tags.append(p_row)
    return true_tags, pred_tags

In [ ]:
# §7 Trainer — head-only, fp32, no AMP; early stopping on val micro-F1
from torch.utils.data import DataLoader, TensorDataset
from transformers import get_linear_schedule_with_warmup

def _loader(cache, bs, shuffle):
    ds = TensorDataset(cache["features"], cache["mask"], cache["labels"])
    return DataLoader(ds, batch_size=bs, shuffle=shuffle)

def _eval(head, cache, bs=64):
    head.eval(); T, P = [], []
    with torch.no_grad():
        for feats, mask, labs in _loader(cache, bs, False):
            feats = feats.float().to(DEVICE); mask = mask.to(DEVICE)
            pred = head(feats, mask)
            pred = pred.cpu().tolist() if head.head_kind == "softmax" else pred
            t, p = decode_predictions(pred, labs.tolist(), head.head_kind); T += t; P += p
    return evaluate_tags(T, P)

def _param_groups(head, lr, crf_lr):
    """CRF transition/start/end matrices need a much higher LR than the emission layer
    to escape the all-O cold start; everything else trains at the base lr."""
    crf_p, base_p = [], []
    for n, p in head.named_parameters():
        (crf_p if n.startswith("crf.") else base_p).append(p)
    groups = [{"params": base_p, "lr": lr}]
    if crf_p:
        groups.append({"params": crf_p, "lr": crf_lr})
    return groups

def train_head(make_head, seed, lr=1e-3, crf_lr=5e-2, epochs=40,
               patience=8, min_epochs=10, bs=64, verbose=False):
    set_seed(seed)
    head = make_head().to(DEVICE)
    opt  = torch.optim.AdamW(_param_groups(head, lr, crf_lr), weight_decay=0.01)
    dl   = _loader(CACHE["train"], bs, True)
    total = len(dl) * epochs
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * total), total)
    best, bad, best_state = -1.0, 0, None
    for ep in range(epochs):
        head.train()
        for feats, mask, labs in dl:
            feats = feats.float().to(DEVICE); mask = mask.to(DEVICE); labs = labs.to(DEVICE)
            loss = head(feats, mask, labs)
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        f1 = _eval(head, CACHE["validation"])["micro_f1"]
        if verbose: print(f"  ep{ep} val micro_f1={f1:.4f}")
        if f1 > best:
            best, bad = f1, 0
            best_state = {k: v.detach().cpu().clone() for k, v in head.state_dict().items()}
        elif ep + 1 >= min_epochs:                 # don't early-stop during CRF cold-start
            bad += 1
            if bad >= patience: break
    head.load_state_dict(best_state)
    return _eval(head, CACHE["test"])

In [ ]:
# §7 (gate) smoke test — CRF head; watch val micro_f1 climb out of the all-O cold start.
# With the higher crf_lr it should pass ~0.3 within a handful of epochs and approach Softmax.
_m = train_head(lambda: CrfHead(H, C), seed=42, epochs=15, verbose=True)
print("smoke test CRF micro_f1 =", round(_m["micro_f1"], 4))
assert _m["micro_f1"] > 0.3, "CRF still under-fitting -> raise crf_lr / epochs further"

In [ ]:
# §8 Run matrix — 3 heads x 3 seeds
import pandas as pd
HEADS = {
    "Softmax": lambda: SoftmaxHead(H, C),
    "CRF":     lambda: CrfHead(H, C),
    "GAT+CRF": lambda: GatCrfHead(H, C),
}
rows = []
for name, mk in HEADS.items():
    for s in SEEDS:
        m = train_head(mk, seed=s)
        rows.append({"head": name, "seed": s,
                     "micro_f1": m["micro_f1"], "macro_f1": m["macro_f1"]})
        print(f'{name} seed{s}: micro={m["micro_f1"]:.4f} macro={m["macro_f1"]:.4f}')
df = pd.DataFrame(rows)
assert (df["micro_f1"] > 0).all()

In [ ]:
# §9 Comparison report — mean +/- std over seeds + per-entity report + CSV
agg = (df.groupby("head")
         .agg(micro_mean=("micro_f1", "mean"), micro_std=("micro_f1", "std"),
              macro_mean=("macro_f1", "mean"), macro_std=("macro_f1", "std"))
         .reindex(["Softmax", "CRF", "GAT+CRF"]).reset_index())
print(agg.to_string(index=False))
_enc_tag = CFG["encoder"].split("/")[-1]
_csv = f'{RESULTS_DIR}/frozen_head_ablation_{_enc_tag}.csv'
agg.to_csv(_csv, index=False)
print(f'\nwrote {_csv}')

for name, mk in HEADS.items():
    best_seed = int(df[df.head == name].sort_values("micro_f1").iloc[-1]["seed"])
    print(f'\n===== {name} (best seed {best_seed}) =====')
    print(train_head(mk, seed=best_seed)["report"])